<a href="https://colab.research.google.com/github/Otza02/land2vec/blob/main/notebooks/prueba_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/Otza02/land2vec.git
%cd /content/land2vec
!pip install -e .

Cloning into 'land2vec'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 93 (delta 35), reused 76 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 4.01 MiB | 10.25 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/land2vec


### ---Restart session before next cell---

In [ ]:
%cd /content/land2vec

In [1]:
import tqdm
from datetime import datetime
import pandas as pd
import os

import torch
from torch.utils.data import DataLoader, random_split
from torch.optim import lr_scheduler

from land2vec.dataset import load_data, SequenceDataset
from land2vec.config import Config
from land2vec.model import GPTDecoder, run_epoch
from land2vec.tokenizer import Tokenizer
from land2vec.utils import get_target_folder, load_config, load_model, load_train_results, save_config, save_model, save_train_results

In [2]:
config = Config(patience=4, device="cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(config.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config.seed)
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
config.device

'cpu'

In [3]:
print("loading data")
file_path = "data/id_seqs_text_2000_2022_chaco_santiago_frontier.zip"
try:
    dataset = load_data(file_path="/content/land2vec/" + file_path, window=config.block_size)
except FileNotFoundError:
    dataset = load_data(file_path="../data/seqs_short.csv", window=config.block_size)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

generator = torch.Generator().manual_seed(config.seed)

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], generator=generator)
loader_kwargs = dict(
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    pin_memory=True,
    persistent_workers=True,
)

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    **loader_kwargs,
)

val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    **loader_kwargs,
)

test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    **loader_kwargs,
)

loading data


100%|██████████| 10/10 [00:00<00:00, 10031.82it/s]


In [4]:
config

Config(block_size=12, n_embd=128, n_head=4, n_layer=4, dropout=0.1, epochs=25, patience=4, lr=0.001, min_lr=1e-06, weight_decay=0.01, num_workers=2, batch_size=256, device='cpu', seed=42)

In [5]:
model = GPTDecoder(
    vocab_size=len(Tokenizer.VOCAB),
    block_size=config.block_size,
    n_embd=config.n_embd,
    n_head=config.n_head,
    n_layer=config.n_layer,
    dropout=config.dropout,
).to(config.device)

optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay, fused=config.device == "cuda")
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs, eta_min=config.min_lr)
scaler = torch.amp.GradScaler(enabled=True) if config.device == "cuda" else None

print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

parameters: 794,880


In [6]:
best_model_state = None
best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "lr": [],
    "epoch_time": [],
    "tokens_per_sec": [],
}

global_start = datetime.now()
for epoch in range(config.epochs):
    start = datetime.now()
    train_loss = run_epoch(model, train_loader, optimizer, config.device, scaler=scaler, use_amp=True)
    val_loss = run_epoch(model, val_loader, optimizer=None, device=config.device, scaler=None, use_amp=True)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_model_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    epoch_seconds = (datetime.now() - start).seconds
    if epoch_seconds < 1:
        epoch_seconds = 1

    tokens_processed = (len(train_loader.dataset) * config.block_size)
    tokens_per_sec = (tokens_processed / epoch_seconds)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["lr"].append(current_lr)
    history["epoch_time"].append(epoch_seconds)
    history["tokens_per_sec"].append(tokens_per_sec)

    ppl = torch.math.exp(train_loss)
    print(
        f"[Epoch {epoch+1:03d}] | "
        f"train={train_loss:.4f} | "
        f"val={val_loss:.4f} | "
        f"ppl={ppl:.3f} | "
        f"lr={current_lr:.2e} | "
        f"time={(epoch_seconds // 60):02}:{(epoch_seconds % 60):02} | "
        f"token/s={tokens_per_sec:,.0f} | "
    )

    if patience_counter >= config.patience:
        print("Early stopping...")
        model.load_state_dict(best_model_state)
        break

total_time = (datetime.now() - global_start).seconds
print("=== Finished Training ===")
print(f"best epoch: {best_epoch}")
print(f"best val loss: {best_val_loss:.4f}")
print(f"total training time: {total_time // 60:02}:{total_time % 60:02}")

c:\Users\Admin\Desktop\unsam\proyecto\land2vec\.venv\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[Epoch 001] | train=3.8895 | val=0.0000 | ppl=48.885 | lr=9.96e-04 | time=00:04 | token/s=24 | 
[Epoch 002] | train=1.6102 | val=2.0557 | ppl=5.004 | lr=9.84e-04 | time=00:01 | token/s=96 | 
[Epoch 003] | train=1.6950 | val=0.0000 | ppl=5.447 | lr=9.65e-04 | time=00:01 | token/s=96 | 
[Epoch 004] | train=0.8794 | val=0.0000 | ppl=2.409 | lr=9.38e-04 | time=00:01 | token/s=96 | 
[Epoch 005] | train=1.2627 | val=0.0000 | ppl=3.535 | lr=9.05e-04 | time=00:01 | token/s=96 | 
Early stopping...
=== Finished Training ===
best epoch: 1
best val loss: 0.0000
total training time: 00:05


In [7]:
# torch.save(model.state_dict(), "first-test.pt")
target_folder = get_target_folder(str(datetime.now().date()))
os.makedirs(str(target_folder), exist_ok=True)
save_config(config, target_folder)
save_model(model, target_folder)
save_train_results(history, target_folder)

Config saved to ..\models\2026-05-19\config.json
Model saved to ..\models\2026-05-19\model.pt
Train data saved to ..\models\2026-05-19\train_data.csv


In [9]:
config = load_config(target_folder)
model = load_model(config, target_folder)
train_results = load_train_results(target_folder)

In [ ]:
model.eval()

correct = 0
total = 0

all_preds = []
all_targets = []

with torch.no_grad():
    for x, y in tqdm.tqdm(test_loader):
        x = x.long().to(device)
        y = y.long().to(device)

        logits, loss = model(x, y)

        preds = torch.argmax(logits, dim=-1)

        correct += (preds == y).sum().item()
        total += y.numel()

        all_preds.append(preds.cpu())
        all_targets.append(y.cpu())

accuracy = correct / total

print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.9953
